In [1]:
import os
import openai
from llama_index.llms.openai import OpenAI
from llama_index.core.node_parser import SentenceWindowNodeParser, SentenceSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.postprocessor import MetadataReplacementPostProcessor
from pprint import pprint

## 관련 논문 읽어오기

In [2]:
# load data
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["papers/modular rag.pdf"]
).load_data()

오늘의 질문 : `"What are three limitations of Naive RAG?" `

In [3]:
question = "What are three limitations of Naive RAG?"

예상 답변 구간 : 

```json
Retrieval Challenges. The retrieval phase often struggles
with precision and recall, leading to the selection of misaligned
or irrelevant chunks, and the missing of crucial information.


Generation Difficulties. In generating responses, the model
may face the issue of hallucination, where it produces content
not supported by the retrieved context. This phase can
also suffer from irrelevance, toxicity, or bias in the outputs,
detracting from the quality and reliability of the responses.


Augmentation Hurdles. Integrating retrieved information
with the different task can be challenging, sometimes resulting
in disjointed or incoherent outputs. The process may also
encounter redundancy when similar information is retrieved
from multiple sources, leading to repetitive responses. Determining
the significance and relevance of various passages and
ensuring stylistic and tonal consistency add further complexity.
Facing complex issues, a single retrieval based on the original
query may not suffice to acquire adequate context information.
Moreover, there’s a concern that generation models might
overly rely on augmented information, leading to outputs that
simply echo retrieved content without adding insightful or
synthesized information.
```

# Config

In [4]:
    # os.environ['OPENAI_API_KEY'] = '...'

llm = OpenAI(model="gpt-4-turbo-preview")
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

## 가장 기본이 되는 base splitter를 활용하여 질문

In [5]:
# sentence splitter
text_splitter = SentenceSplitter(separator='. ', chunk_size=200, chunk_overlap=0)

# node 생성
base_nodes = text_splitter.get_nodes_from_documents(documents)

# 임시 index 생성
base_index = VectorStoreIndex(base_nodes)

In [6]:
query_engine = base_index.as_query_engine(similarity_top_k=2)
vector_response = query_engine.query(
    question
)

In [7]:
pprint(vector_response.response)

('Three limitations of Naive RAG are its lack of optimization strategies, its '
 'reliance on a chain-like structure, and its limited flexibility compared to '
 'Advanced RAG and Modular RAG.')


In [8]:
print(vector_response.source_nodes[0].node.text)

RAG bridges this information
gap by sourcing and incorporating knowledge from external
databases. In this case, it gathers relevant news articles related
to the user’s query. These articles, combined with the original
question, form a comprehensive prompt that empowers LLMs
to generate a well-informed answer.
The RAG research paradigm is continuously evolving, and
we categorize it into three stages: Naive RAG, Advanced
RAG, and Modular RAG, as showed in Figure 3. Despite
RAG method are cost-effective and surpass the performance
of the native LLM, they also exhibit several limitations.
The development of Advanced RAG and Modular RAG is
a response to these specific shortcomings in Naive RAG.
A. Naive RAG
The Naive RAG research paradigm represents the earli-
est methodology, which gained prominence shortly after the


In [9]:
print(vector_response.source_nodes[1].node.text)

4
Fig. 3. Comparison between the three paradigms of RAG. (Left) Naive RAG mainly consists of three parts: indexing, retrieval and generation. (Middle)
Advanced RAG proposes multiple optimization strategies around pre-retrieval and post-retrieval, with a process similar to the Naive RAG, still following a
chain-like structure. (Right) Modular RAG inherits and develops from the previous paradigm, showcasing greater flexibility overall. This is evident in the
introduction of multiple specific functional modules and the replacement of existing modules. The overall process is not limited to sequential retrieval and
generation; it includes methods such as iterative and adaptive retrieval.
Pre-retrieval process . In this stage, the primary focus is
on optimizing the indexing structure and the original query.


## Sliding window parser를 활용하여 질문

In [10]:
# sentence splitter
text_splitter = SentenceSplitter(separator='. ', chunk_size=200, chunk_overlap=100) # chunk overlap을 활용하여 주변 컨텍스트까지 고려 하도록 함

# node 생성
base_nodes = text_splitter.get_nodes_from_documents(documents)

# 임시 index 생성
base_index = VectorStoreIndex(base_nodes)

In [11]:
print(base_nodes[0].text)

1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, Jinliu Panb, Yuxi Bic, Yi Daia, Jiawei Suna, Meng
Wangc, and Haofen Wanga,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University
cCollege of Design and Innovation, Tongji University
Abstract —Large Language Models (LLMs) showcase impres-
sive capabilities but encounter challenges like hallucination,
outdated knowledge, and non-transparent, untraceable reasoning
processes. Retrieval-Augmented Generation (RAG) has emerged
as a promising solution by incorporating knowledge from external
databases.


In [12]:
print(base_nodes[1].text)

Retrieval-Augmented Generation (RAG) has emerged
as a promising solution by incorporating knowledge from external
databases. This enhances the accuracy and credibility of the
generation, particularly for knowledge-intensive tasks, and allows
for continuous knowledge updates and integration of domain-
specific information. RAG synergistically merges LLMs’ intrin-
sic knowledge with the vast, dynamic repositories of external
databases. This comprehensive review paper offers a detailed
examination of the progression of RAG paradigms, encompassing
the Naive RAG, the Advanced RAG, and the Modular RAG.
It meticulously scrutinizes the tripartite foundation of RAG
frameworks, which includes the retrieval, the generation and the
augmentation techniques.


In [13]:
query_engine = base_index.as_query_engine(similarity_top_k=2)
vector_response = query_engine.query(
    question
)
print(vector_response)

The limitations of Naive RAG include retrieval challenges, generation difficulties, and augmentation hurdles.


In [14]:
print(vector_response.source_nodes[0].node.text)

The RAG research paradigm is continuously evolving, and
we categorize it into three stages: Naive RAG, Advanced
RAG, and Modular RAG, as showed in Figure 3. Despite
RAG method are cost-effective and surpass the performance
of the native LLM, they also exhibit several limitations.
The development of Advanced RAG and Modular RAG is
a response to these specific shortcomings in Naive RAG.
A. Naive RAG
The Naive RAG research paradigm represents the earli-
est methodology, which gained prominence shortly after the


In [15]:
print(vector_response.source_nodes[1].node.text)

In cases of ongoing
dialogues, any existing conversational history can be integrated
into the prompt, enabling the model to engage in multi-turn
dialogue interactions effectively.
However, Naive RAG encounters notable drawbacks:Retrieval Challenges . The retrieval phase often struggles
with precision and recall, leading to the selection of misaligned
or irrelevant chunks, and the missing of crucial information.
Generation Difficulties . In generating responses, the model
may face the issue of hallucination, where it produces con-
tent not supported by the retrieved context. This phase can
also suffer from irrelevance, toxicity, or bias in the outputs,
detracting from the quality and reliability of the responses.
Augmentation Hurdles . Integrating retrieved information
with the different task can be challenging, sometimes resulting
in disjointed or incoherent outputs.


## Small to big을 활용한 RAG

In [16]:
# window splitter
node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text",
)

# node 생성
window_nodes = node_parser.get_nodes_from_documents(documents)

# 임시 index 생성
window_index = VectorStoreIndex(window_nodes)

In [17]:
query_engine = window_index.as_query_engine(
    similarity_top_k=2,
    # metat data replacement. 하나의 문장
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)

vector_response = query_engine.query(
    question
)
print(vector_response)

The limitations of Naive RAG include struggles with precision and recall during the retrieval phase, leading to the selection of misaligned or irrelevant information, difficulties in generating responses that may result in producing content not supported by the retrieved context (hallucination), and the direct reliance on the user's original query for retrieval, which can be challenging due to the complexity and ambiguity of language.


In [20]:
question

'What are three limitations of Naive RAG?'

In [18]:
print("Searched sentence : \n", vector_response.source_nodes[0].node.metadata['original_text'])
print("\n", "-"*60, "\n")
print("Retrieved sentence : \n", vector_response.source_nodes[0].node.metadata['window'])

Searched sentence : 
 However, Naive RAG encounters notable drawbacks:Retrieval Challenges . 

 ------------------------------------------------------------ 

Retrieved sentence : 
 The posed query and selected documents are
synthesized into a coherent prompt to which a large language
model is tasked with formulating a response.  The model’s
approach to answering may vary depending on task-specific
criteria, allowing it to either draw upon its inherent parametric
knowledge or restrict its responses to the information con-
tained within the provided documents.  In cases of ongoing
dialogues, any existing conversational history can be integrated
into the prompt, enabling the model to engage in multi-turn
dialogue interactions effectively.
 However, Naive RAG encounters notable drawbacks:Retrieval Challenges .  The retrieval phase often struggles
with precision and recall, leading to the selection of misaligned
or irrelevant chunks, and the missing of crucial information.
 Generation Diff

In [19]:
print("Searched sentence : \n", vector_response.source_nodes[1].node.metadata['original_text'])
print("\n", "-"*60, "\n")
print("Retrieved sentence : \n", vector_response.source_nodes[1].node.metadata['window'])

Searched sentence : 
 C. Query Optimization
One of the primary challenges with Naive RAG is its
direct reliance on the user’s original query as the basis for
retrieval. 

 ------------------------------------------------------------ 

Retrieved sentence : 
 Another advantage is the transformation of the
information retrieval process into instructions that LLM can
comprehend, thereby enhancing the accuracy of knowledge
retrieval and enabling LLM to generate contextually coherent
responses, thus improving the overall efficiency of the RAG
system.  To capture the logical relationship between document
content and structure, KGP [91] proposed a method of building
an index between multiple documents using KG.  This KG
consists of nodes (representing paragraphs or structures in the
documents, such as pages and tables) and edges (indicating
semantic/lexical similarity between paragraphs or relationships
within the document structure), effectively addressing knowl-
edge retrieval and reasoning 

## 문장들의 의미를 기반으로 '편집점'을 찾는 semantic chunking

In [21]:
# semantic splitter
semantic_splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)

# node 생성
semantic_nodes = semantic_splitter.get_nodes_from_documents(documents)

# 임시 index 생성
semantic_index = VectorStoreIndex(semantic_nodes)

In [22]:
print(semantic_nodes[0].get_content())

1
Retrieval-Augmented Generation for Large
Language Models: A Survey
Yunfan Gaoa, Yun Xiongb, Xinyu Gaob, Kangxiang Jiab, Jinliu Panb, Yuxi Bic, Yi Daia, Jiawei Suna, Meng
Wangc, and Haofen Wanga,c
aShanghai Research Institute for Intelligent Autonomous Systems, Tongji University
bShanghai Key Laboratory of Data Science, School of Computer Science, Fudan University
cCollege of Design and Innovation, Tongji University
Abstract —Large Language Models (LLMs) showcase impres-
sive capabilities but encounter challenges like hallucination,
outdated knowledge, and non-transparent, untraceable reasoning
processes. Retrieval-Augmented Generation (RAG) has emerged
as a promising solution by incorporating knowledge from external
databases. This enhances the accuracy and credibility of the
generation, particularly for knowledge-intensive tasks, and allows
for continuous knowledge updates and integration of domain-
specific information. RAG synergistically merges LLMs’ intrin-
sic knowledge with th

In [23]:
print(semantic_nodes[1].get_content())

At the end, this article delineates
the challenges currently faced and points out prospective avenues
for research and development1.
Index Terms —Large language model, retrieval-augmented gen-
eration, natural language processing, information retrieval
I. I NTRODUCTION
LARGE language models (LLMs) have achieved remark-
able success, though they still face significant limitations,
especially in domain-specific or knowledge-intensive tasks [1],
notably producing “hallucinations” [2] when handling queries
beyond their training data or requiring current information. To
overcome challenges, Retrieval-Augmented Generation (RAG)
enhances LLMs by retrieving relevant document chunks from
external knowledge base through semantic similarity calcu-
lation. 


![alt text](semantic_chunking.png "Title")

source : https://www.youtube.com/watch?v=8OJC21T2SL4&t=1933s

---

## HYDE

In [26]:
question

'What are three limitations of Naive RAG?'

In [24]:
hyde = HyDEQueryTransform(include_original=True)

transformed = hyde(question)

In [25]:
for i in transformed.custom_embedding_strs:
    pprint(i)
    print()

('One limitation of Naive RAG is its inability to handle complex relationships '
 'between entities. The algorithm assumes that all relationships are '
 'independent, which may not always be the case in real-world scenarios where '
 'entities can have multiple interconnected relationships. Another limitation '
 'is its reliance on a fixed threshold for determining the strength of '
 'relationships, which may not be suitable for all datasets as the optimal '
 'threshold can vary depending on the context. Additionally, Naive RAG does '
 'not take into account the directionality of relationships, treating all '
 'relationships as bidirectional when in reality, some relationships may be '
 'unidirectional. This can lead to inaccuracies in the analysis of the network '
 'structure. Overall, while Naive RAG is a simple and easy-to-implement '
 'algorithm, it may not be suitable for complex datasets with intricate '
 'relationships between entities.')

'What are three limitations of Naive RAG